# Feature Engineering

Perkiraan waktu: **45** menit

Kualitas fitur menentukan seberapa baik model machine learning bekerja. Proses penyusunan fitur tersebut disebut feature engineering, dan mencakup tiga tahap: transformasi fitur (mengubah fitur asli), seleksi fitur (memilih fitur yang paling berguna), serta ekstraksi fitur (menggabungkan fitur lama menjadi fitur baru yang lebih informatif).

## Tujuan

Setelah menyelesaikan latihan ini, kamu dapat:

- Memahami jenis-jenis feature engineering
  - Transformasi fitur
    - Penanganan variabel kategorikal: one hot encoding dan label encoding
    - Transformasi tanggal dan waktu
  - Seleksi fitur
  - Ekstraksi fitur dengan Principal Component Analysis


## Persiapan

Pustaka yang dipakai:

- `pandas` untuk mengelola tabel data.
- `numpy` untuk operasi matematika.
- `seaborn` dan `matplotlib` untuk visualisasi.
- `plotly.express` untuk grafik interaktif.
- `scikit-learn` untuk penskalaan dan PCA.

Berkas `airlines_data.xlsx` dibaca lewat bantuan `openpyxl`, jadi pastikan paket tersebut tersedia di environment-mu.


In [ ]:
# Imports
import urllib.request
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")


## Membaca dan memahami data

Data yang dipakai adalah `airlines_data.xlsx`, berisi harga tiket pesawat berbagai maskapai pada Maret sampai Juni 2019 untuk sejumlah rute. Harga tiket dipengaruhi banyak faktor, seperti nama maskapai, tanggal perjalanan, rute, jam berangkat dan tiba, kota asal dan tujuan, serta durasi penerbangan.


In [ ]:
# Unduh berkas sekali saja, lalu muat ke DataFrame
data_path = Path("airlines_data.xlsx")
data_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML0232EN-SkillsNetwork/asset/airlines_data.xlsx"

if not data_path.exists():
    urllib.request.urlretrieve(data_url, data_path)

flight_data = pd.read_excel(data_path)
flight_data.head()


Tipe data setiap kolom diperiksa dengan `info()`. Hampir semua kolom bertipe teks, kecuali `Price` yang berupa bilangan bulat.


In [ ]:
flight_data.info()


Statistik kolom numerik, dalam hal ini `Price`, dilihat lewat `describe()`.


In [ ]:
flight_data.describe()


Nilai kosong dihitung per kolom. Hasilnya menunjukkan satu baris tanpa `Route` dan `Total_Stops`. Baris seperti ini bisa dibuang atau diisi. Di sini nilainya diteruskan dari baris sebelumnya memakai `ffill`.


In [ ]:
flight_data.isnull().sum()


In [ ]:
flight_data = flight_data.ffill()
flight_data.isnull().sum().sum()


## Transformasi fitur

Transformasi fitur berarti mengubah fitur asli menjadi bentuk lain, misalnya encoding, penskalaan, atau diskretisasi (mengubah variabel kontinu menjadi beberapa interval).

### Variabel kategorikal

Variabel kategorikal menyimpan data kualitatif yang tidak punya makna matematis langsung, sehingga harus diubah menjadi angka sebelum dipakai model. Kolom `Airline` diperiksa lebih dulu untuk melihat daftar kategorinya.


In [ ]:
flight_data["Airline"].unique().tolist()


Beberapa maskapai tercatat dalam dua versi, misalnya `Jet Airways` dan `Jet Airways Business`. Varian seperti ini digabung ke kategori induknya agar konsisten.


In [ ]:
# Gabungkan varian maskapai ke kategori induk
airline_aliases = {
    "Vistara Premium economy": "Vistara",
    "Jet Airways Business": "Jet Airways",
}
flight_data["Airline"] = flight_data["Airline"].replace(airline_aliases)
flight_data["Airline"].unique().tolist()


## Latihan 1

Masih ada satu varian yang belum digabung, yaitu `Multiple carriers Premium economy`. Gabungkan kategori tersebut ke `Multiple carriers` memakai `np.where()`, lalu cetak daftar maskapai dengan `unique().tolist()`.


In [ ]:
flight_data["Airline"] = np.where(
    flight_data["Airline"] == "Multiple carriers Premium economy",
    "Multiple carriers",
    flight_data["Airline"],
)
flight_data["Airline"].unique().tolist()


Sekarang tersisa sembilan maskapai, masing-masing dengan satu label konsisten. Sebaran jumlah penerbangannya diperiksa pada latihan berikut.

#### One hot encoding

Salah satu cara mengubah kategori menjadi angka adalah one hot encoding. Setiap nilai kategori dibuatkan satu kolom biner. Di sini kolom `Airline`, `Source`, dan `Destination` diubah memakai `get_dummies()`.


In [ ]:
encoded_data = pd.get_dummies(
    flight_data, columns=["Airline", "Source", "Destination"]
)
encoded_data.head()


Jumlah kolom sebelum dan sesudah encoding dibandingkan untuk melihat dampaknya.


In [ ]:
print(f"Sebelum encoding: {flight_data.shape}")
print(f"Sesudah encoding: {encoded_data.shape}")


Dari 11 kolom, jumlahnya menjadi 28. Setiap kategori unik pada kolom kategorikal menghasilkan satu kolom baru berisi 1 bila kategori itu muncul pada baris tersebut dan 0 bila tidak.


## Latihan 2

Gunakan `value_counts()` untuk melihat sebaran nilai pada kolom `Total_Stops`.


In [ ]:
flight_data["Total_Stops"].value_counts()


Sebagian besar penerbangan transit satu kali, disusul penerbangan tanpa transit. Penerbangan dengan tiga atau empat transit sangat jarang.

#### Label encoding

`Total_Stops` sebenarnya kategorikal dengan urutan alami, sehingga cukup diubah menjadi angka 0 sampai 4 memakai `replace()`.


In [ ]:
stop_mapping = {"non-stop": 0, "1 stop": 1, "2 stops": 2, "3 stops": 3, "4 stops": 4}
encoded_data["Total_Stops"] = encoded_data["Total_Stops"].replace(stop_mapping).astype(int)
encoded_data["Total_Stops"].value_counts().sort_index()


### Transformasi tanggal dan waktu

#### Kolom durasi

`Duration` menyimpan lama penerbangan dalam bentuk teks, misalnya `2h 50m`, `19h`, atau `5m`. Nilai seperti ini belum bisa dipakai model. Pola jam dan menit diekstrak dengan regex, lalu diubah menjadi menit total.


In [ ]:
# Pisahkan jam dan menit dari teks durasi
duration_parts = encoded_data["Duration"].str.extract(r"(?:(\d+)h)?\s*(?:(\d+)m)?")
duration_hours = duration_parts[0].fillna(0).astype(int)
duration_minutes = duration_parts[1].fillna(0).astype(int)

encoded_data["Duration_hours"] = duration_hours
encoded_data["Duration_minutes"] = duration_minutes
encoded_data["Duration_Total_mins"] = duration_hours * 60 + duration_minutes
encoded_data[["Duration", "Duration_hours", "Duration_minutes", "Duration_Total_mins"]].head()


Tiga kolom numerik baru terbentuk. `Duration_Total_mins` menyatukan jam dan menit ke satu satuan sehingga lebih praktis dipakai pada analisis regresi.

#### Kolom jam keberangkatan dan kedatangan

`Dep_Time` dan `Arrival_Time` diubah dengan `to_datetime()`, lalu jam dan menitnya diambil lewat `.dt.hour` dan `.dt.minute`. Sebagian nilai `Arrival_Time` menyertakan tanggal kedatangan, sehingga hanya lima karakter pertama yang diambil.


In [ ]:
departure_clock = pd.to_datetime(encoded_data["Dep_Time"], format="%H:%M")
encoded_data["Dep_Hour"] = departure_clock.dt.hour
encoded_data["Dep_Min"] = departure_clock.dt.minute
encoded_data[["Dep_Time", "Dep_Hour", "Dep_Min"]].head()


## Latihan 3

Lakukan hal yang sama untuk `Arrival_Time`: ambil jam dan menitnya, lalu simpan sebagai `Arrival_Hour` dan `Arrival_Min`.


In [ ]:
arrival_clock = pd.to_datetime(encoded_data["Arrival_Time"].str.slice(0, 5), format="%H:%M")
encoded_data["Arrival_Hour"] = arrival_clock.dt.hour
encoded_data["Arrival_Min"] = arrival_clock.dt.minute
encoded_data[["Arrival_Time", "Arrival_Hour", "Arrival_Min"]].head()


#### Pembagian waktu menjadi zona waktu

Jam keberangkatan dan kedatangan dapat dikelompokkan menjadi empat zona: malam, pagi, siang, dan sore. Pengelompokan memakai `cut()` dengan batas 0, 6, 12, 18, dan 24. Parameter `include_lowest=True` dipakai agar jam 00.00 tetap masuk kategori malam.


In [ ]:
time_bins = [0, 6, 12, 18, 24]
time_labels = ["Night", "Morning", "Afternoon", "Evening"]

encoded_data["dep_timezone"] = pd.cut(
    encoded_data["Dep_Hour"], time_bins, labels=time_labels, include_lowest=True
)
encoded_data["dep_timezone"].value_counts().sort_index()


## Latihan 4

Ubah `Arrival_Hour` menjadi zona waktu dengan cara yang sama, lalu simpan hasilnya pada kolom `arr_timezone`.


In [ ]:
encoded_data["arr_timezone"] = pd.cut(
    encoded_data["Arrival_Hour"], time_bins, labels=time_labels, include_lowest=True
)
encoded_data["arr_timezone"].value_counts().sort_index()


Keberangkatan pagi paling banyak terjadi, sedangkan penerbangan malam paling sedikit. Pola semacam ini bisa menjadi sinyal tambahan bagi model.

#### Kolom tanggal perjalanan

`Date_of_Journey` masih berupa teks, jadi bagian bulan, hari, tahun, dan nama hari diekstrak menjadi kolom terpisah. Format tanggalnya hari/bulan/tahun, sehingga `dayfirst=True` dipakai saat konversi.


In [ ]:
journey_date = pd.to_datetime(encoded_data["Date_of_Journey"], dayfirst=True)
encoded_data["Month"] = journey_date.dt.month
encoded_data[["Date_of_Journey", "Month"]].head()


## Latihan 5

Buat kolom `Day` dan `Year` dari `journey_date` dengan cara yang sama, lalu tambahkan kolom `day_of_week` berisi nama hari memakai `dt.day_name()`.


In [ ]:
encoded_data["Day"] = journey_date.dt.day
encoded_data["Year"] = journey_date.dt.year
encoded_data["day_of_week"] = journey_date.dt.day_name()
encoded_data[["Date_of_Journey", "Day", "Month", "Year", "day_of_week"]].head()


Harga tiket umumnya berbeda antara hari kerja dan akhir pekan, sehingga nama hari berguna sebagai fitur kategorikal tambahan.

## Seleksi fitur

Tahap ini memilih atribut yang paling menjelaskan hubungan dengan target `Price`. Kolom yang tidak relevan seperti `Route`, `Additional_Info`, dan kolom waktu asli dikeluarkan. Kolom dummy diubah ke bilangan bulat lebih dulu agar ikut terhitung sebagai numerik.


In [ ]:
# Kolom dummy hasil get_dummies bertipe bool, ubah menjadi bilangan bulat
bool_columns = encoded_data.select_dtypes(include="bool").columns
encoded_data[bool_columns] = encoded_data[bool_columns].astype(int)

selected_data = encoded_data.select_dtypes(include=np.number)
selected_data.columns.tolist()


Hubungan antar variabel terpilih digambarkan sebagai heatmap korelasi.


In [ ]:
plt.figure(figsize=(18, 18))
sns.heatmap(selected_data.corr(), annot=True, cmap="RdYlGn")
plt.show()


Pada heatmap, warna hijau menandai korelasi positif kuat (dua variabel bergerak searah), sedangkan warna merah menandai korelasi negatif (satu naik, yang lain turun).


In [ ]:
# Korelasi setiap fitur terhadap Price, diurutkan dari negatif ke positif
price_correlation = selected_data.corr()["Price"].sort_values()
price_correlation


Koefisien korelasi juga bisa digambarkan sebagai diagram batang agar lebih mudah dibaca.


In [ ]:
price_correlation.plot(kind="bar", figsize=(10, 8))
plt.show()


`Total_Stops` punya korelasi positif paling kuat terhadap harga, disusul `Duration_Total_mins`. Beberapa maskapai seperti IndiGo dan SpiceJet berkorelasi negatif, artinya tiketnya cenderung lebih murah. Fitur dengan korelasi mendekati nol bisa dipertimbangkan untuk dibuang.

## Ekstraksi fitur dengan Principal Component Analysis (opsional)

### PCA dengan scikit-learn

Reduksi dimensi menggabungkan fitur lama menjadi fitur baru yang lebih ringkas tanpa kehilangan terlalu banyak informasi. PCA mencari bidang yang paling dekat dengan sebaran data, lalu memproyeksikan data ke bidang tersebut sehingga beberapa dimensi menyatu.

Sebelum PCA, data diskalakan memakai `StandardScaler()` karena PCA peka terhadap perbedaan skala antar fitur.


In [ ]:
# Pisahkan fitur dan target
features = selected_data.drop(columns=["Price"])
target = selected_data["Price"]

features_scaled = StandardScaler().fit_transform(features)
features_scaled.shape


Setelah data diskalakan, `fit_transform()` diterapkan untuk mereduksi data menjadi dua dimensi.


In [ ]:
pca = PCA(n_components=2)
pca.fit_transform(features_scaled)


### Explained variance ratio

`explained_variance_ratio_` menunjukkan porsi varians data yang diwakili setiap komponen utama.


In [ ]:
explained_variance = pca.explained_variance_ratio_
explained_variance


Komponen pertama menyerap 17.55% varians dan komponen kedua 12.11%, sehingga keduanya bersama-sama baru mencakup sekitar 29.66%. Informasi pada data ini tersebar di banyak dimensi.


## Latihan 6 (opsional)

Coba variasikan jumlah komponen untuk melihat berapa dimensi yang dibutuhkan agar sebagian besar variabilitas tetap terjaga. Gambarkan juga varians setiap komponen dengan diagram batang.


In [ ]:
pca_seven = PCA(n_components=7)
pca_seven.fit_transform(features_scaled)
seven_variance = pca_seven.explained_variance_ratio_
seven_variance


In [ ]:
with plt.style.context("dark_background"):
    figure, axis = plt.subplots(figsize=(6, 4))
    axis.bar(
        range(len(seven_variance)),
        seven_variance,
        alpha=0.5,
        align="center",
        label="individual explained variance",
    )
    axis.set_ylabel("Explained variance ratio")
    axis.set_xlabel("Principal components")
    axis.legend(loc="best")
    figure.tight_layout()
    plt.show()


### Memilih jumlah dimensi yang tepat

Daripada menentukan jumlah komponen secara sembarang, lebih mudah memilih jumlah yang membuat total varians mencapai ambang tertentu, misalnya 95%. Kode berikut menjalankan PCA tanpa reduksi, lalu menghitung jumlah komponen minimum untuk mencapai ambang tersebut.


In [ ]:
full_pca = PCA()
full_pca.fit(features_scaled)
cumulative_variance = np.cumsum(full_pca.explained_variance_ratio_)
components_needed = np.argmax(cumulative_variance >= 0.95) + 1
components_needed


Dibutuhkan 16 komponen untuk mencapai 95% varians. Angka tersebut bisa langsung dipakai sebagai `n_components`, tetapi ada cara lebih ringkas: isi `n_components` dengan bilangan pecahan antara 0.0 dan 1.0 yang menyatakan porsi varians yang ingin dipertahankan.


In [ ]:
pca_95 = PCA(n_components=0.95)
features_reduced = pca_95.fit_transform(features_scaled)
features_reduced.shape


Jumlah komponen juga bisa ditentukan secara grafis lewat kurva varians kumulatif. Titik belok pada kurva, saat pertambahan varians mulai melambat, biasanya menjadi pilihan jumlah komponen yang wajar.


In [ ]:
px.area(
    x=range(1, cumulative_variance.shape[0] + 1),
    y=cumulative_variance,
    labels={"x": "# Components", "y": "Explained Variance"},
)
